Goal: We want to extract ONLY the residential/commercial premises ('RCN_Lokal') 
from the GML file, convert the nested XML structure into a flat Pandas DataFrame
and inspect the data structure to prepare for the Data Cleaning phase.

In [24]:
import pandas as pd
import xml.etree.ElementTree as ET

In [18]:
file_path = '../data/Baza danych RCN Poznań 2021-2025.gml'

**Task 1: Load the XML tree and define the dictionary of namespaces we will need.**

1. Use ET.parse() to load the file and get the root element.
2. Define a dictionary named 'namespaces' with 'gml' and 'rcn' prefixes.

In [ ]:
# SOLUTION
tree = ET.parse(file_path)
root = tree.getroot()

namespaces = {
    'gml': 'http://www.opengis.net/gml/3.2',
    'rcn': 'urn:gugik:specyfikacje:gmlas:rejestrcennieruchomosci:1.0',
    'xlink': 'http://www.w3.org/1999/xlink' 
}
print(f"Successfully loaded XML. Root tag: {root.tag.split('}')[-1]}")

Successfully loaded XML. Root tag: FeatureCollection


**Task 2: Find all elements matching the 'rcn:RCN_Lokal' tag within the document.**

1. Use root.findall() with the correct XPath and namespaces to find all 'RCN_Lokal' nodes.
2. Count how many premises were found.

In [ ]:
# SOLUTION
lokale_nodes = root.findall('.//rcn:RCN_Lokal', namespaces)
print(f"Found {len(lokale_nodes)} premises (lokale) in the dataset.")

Found 65461 premises (lokale) in the dataset.


**Task 3: Flattening the Nested Data**

1. Create an empty list `lokale_data`.
2. Loop through `lokale_nodes`. For each node, loop through its children.
3. Extract the tag name (ignoring the URI namespace) and its text value.
4. Append the created dictionary to the list.

In [ ]:
# SOLUTION
lokale_data = []

for node in lokale_nodes:
    lokal_dict = {}
    for child in node:
        tag_name = child.tag.split('}')[-1] 
        
        value = child.text.strip() if child.text and child.text.strip() else None
        if not value and child.attrib:
            value = child.attrib.get(f"{{{namespaces['xlink']}}}href", None)
            
        lokal_dict[tag_name] = value
        
    lokale_data.append(lokal_dict)

print(f"Successfully flattened {len(lokale_data)} records. First record preview:")
print(lokale_data[0] if lokale_data else "No data found.")

Successfully flattened 65461 records. First record preview:
{'boundedBy': None, 'idLokalu': '306401_1.0005.AR_03.84.1_BUD.291_LOK', 'georeferencja': None, 'funkcjaLokalu': '1', 'liczbaIzb': '3', 'nrKondygnacji': '5', 'powUzytkowaLokalu': '61.87', 'cenaLokaluBrutto': '400000.0', 'adresBudynkuZLokalem': None}


**Task 4: Converting it into a tabular format.**

1. Convert the `lokale_data` list into a Pandas DataFrame called `df_lokale`.
2. Display the first 5 rows.

In [ ]:
# SOLUTION 
df_lokale = pd.DataFrame(lokale_data)
print("\nDataFrame Head:")
print(df_lokale.head())


DataFrame Head:
  boundedBy                              idLokalu georeferencja funkcjaLokalu  \
0      None  306401_1.0005.AR_03.84.1_BUD.291_LOK          None             1   
1      None  306401_1.0005.AR_03.84.1_BUD.392_LOK          None             6   
2      None  306401_1.0039.AR_32.130.3_BUD.32_LOK          None             1   
3      None  306401_1.0039.AR_32.130.3_BUD.19_LOK          None             1   
4      None  306401_1.0039.AR_32.130.3_BUD.34_LOK          None             1   

  liczbaIzb nrKondygnacji powUzytkowaLokalu cenaLokaluBrutto  \
0         3             5             61.87         400000.0   
1         5            -3          11475.43          50000.0   
2         2             5             48.26         267000.0   
3         3             5             88.91         448995.0   
4         2             1             57.26         300000.0   

   adresBudynkuZLokalem powUzytkowaPomieszczenPrzynal dodatkoweInformacje  \
0                   NaN           

**Task 5: Inspect the data types and check for missing values (NaNs).**

1. Use .info() to check column data types.
2. Use .isnull().sum() to count missing values in each column.

In [27]:
# SOLUTION 
print("\n Data Info")
df_lokale.info() 

print("\n Missing Values Count")
missing_values = df_lokale.isnull().sum()
print(missing_values[missing_values > 0])


 Data Info
<class 'pandas.DataFrame'>
RangeIndex: 65461 entries, 0 to 65460
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   boundedBy                      0 non-null      object 
 1   idLokalu                       65461 non-null  str    
 2   georeferencja                  0 non-null      object 
 3   funkcjaLokalu                  65461 non-null  str    
 4   liczbaIzb                      50514 non-null  str    
 5   nrKondygnacji                  62888 non-null  str    
 6   powUzytkowaLokalu              65461 non-null  str    
 7   cenaLokaluBrutto               63401 non-null  str    
 8   adresBudynkuZLokalem           0 non-null      float64
 9   powUzytkowaPomieszczenPrzynal  11247 non-null  str    
 10  dodatkoweInformacje            1080 non-null   str    
 11  kwotaPodatkuVAT                145 non-null    str    
dtypes: float64(1), object(2), str(9)
memory usage